<a href="https://colab.research.google.com/github/jianchang512/AIGenerateTool/blob/main/AIGenerateToolColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run AIGenerateTool WebUI on Google Colab

This notebook installs [AIGenerateTool](https://github.com/jianchang512/AIGenerateTool) in an isolated Python 3.10 environment and launches its Gradio WebUI. Colab runtimes are temporary, so generated files and configuration are removed when the runtime is reset.

> **Note:** The WebUI only exposes a subset of AIGenerateTool' features (basic video translation, subtitle recognition/translation, TTS). For the full feature set (voice cloning, real-time editing, more API channels), run the desktop client (`sp.py`) locally instead. See [docs/webui.md](docs/webui.md) for details.

## 1. Install AIGenerateTool

The setup is safe to run again: it clones the repository on the first run and updates it on later runs. Dependencies are installed in the project's virtual environment (via `uv`) to avoid conflicts with Colab's preinstalled packages. `AIGenerateTool` requires Python 3.10 specifically.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/vancongquoc121/AIGenerateTool02.git"
REPO_DIR = Path("/content/AIGenerateTool02")

# System packages required by AIGenerateTool (ffmpeg for media processing, libsndfile for audio I/O).
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "libsndfile1-dev"], check=True)

# Update an existing checkout so rerunning this cell does not fail during clone.
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.10"], check=True)
# Use the lockfile in an isolated environment without changing Colab's preinstalled packages.
subprocess.run(["uv", "sync", "--frozen", "--extra", "webui", "--python", "3.10"], check=True)
print(f"AIGenerateTool is ready in {REPO_DIR}")

## 2. Launch the WebUI

`webui.py` is built on Gradio, which can create its own public link via `--share`, so no separate tunnel service (e.g. ngrok) is needed. The cell waits for the `*.gradio.live` link to appear in the server log before printing it. If startup fails, it includes the recent server log in the error.

After opening the URL, use the **⚙️ 渠道设置** (Channel Settings) tab to configure the required model and media API keys.

In [ ]:
import re
import time

# Must match the --port value hardcoded in webui.sh.
PORT = 7861
LOG_PATH = Path("/content/AIGenerateTool-webui.log")
PUBLIC_URL_RE = re.compile(r"https://[\w-]+\.gradio\.live")

# Make this cell safe to rerun by closing the previous process and log handle.
previous_webui_proc = globals().get("webui_proc")
if previous_webui_proc is not None and previous_webui_proc.poll() is None:
    previous_webui_proc.terminate()
    previous_webui_proc.wait(timeout=10)
previous_webui_log = globals().get("webui_log")
if previous_webui_log is not None and not previous_webui_log.closed:
    previous_webui_log.close()

webui_log = LOG_PATH.open("w", encoding="utf-8")
webui_proc = subprocess.Popen(
    ["bash", "webui.sh"],
    cwd=REPO_DIR,
    stdout=webui_log,
    stderr=subprocess.STDOUT,
    text=True,
)

# Poll the log for the Gradio public link; first-time model/library imports can take a while on Colab CPU,
# and creating the `--share` tunnel itself can take a while.
deadline = time.time() + 600
last_size = 0
public_url = None
while time.time() < deadline:
    exited = webui_proc.poll() is not None
    webui_log.flush()
    log_text = LOG_PATH.read_text(encoding="utf-8", errors="replace")
    match = PUBLIC_URL_RE.search(log_text)
    if match:
        public_url = match.group(0)
        break
    # Surface new log output while waiting so long tunnel setups are not silent.
    if len(log_text) > last_size:
        print(log_text[last_size:], end="")
        last_size = len(log_text)
    if exited:
        break
    time.sleep(2)

if not public_url:
    webui_log.flush()
    exit_code = webui_proc.poll()
    status = "timed out while the process was still running" if exit_code is None else f"process exited with code {exit_code}"
    recent_log = LOG_PATH.read_text(encoding="utf-8", errors="replace")[-2000:]
    raise RuntimeError(f"WebUI failed to start ({status}). Recent log:\n{recent_log}")

print("AIGenerateTool WebUI is ready:")
print(public_url)
print(f"Server log: {LOG_PATH}")
